<a href="https://colab.research.google.com/github/Baasmaala/road-safety-explorer/blob/basmala/notebooks/%2001_data_cleaningclustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/baasmaala/road-safety-explorer.git
%cd road-safety-explorer

Cloning into 'road-safety-explorer'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 98 (delta 42), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 264.64 KiB | 6.62 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/road-safety-explorer


# Notebook 01 — Cleaning the WHO data and finding country clusters

This is the first notebook in the pipeline. It takes the raw WHO Global Status Report on Road Safety 2023 spreadsheet and turns it into something the rest of the project can actually use.

Two things happen here:

1. **Cleaning.** The raw file has two header rows, inconsistent country names, and a lot of columns we don't need. We sort that out.
2. **Clustering.** Once the country-level features are clean, we run K-Means to group countries into road-safety profiles. The output of this notebook feeds directly into my teammates Omar's PCA notebook (`02_pca_anomaly.ipynb`).

Palestine appears in the WHO file as *"occupied Palestinian territory, including east Jerusalem"*. We rename it to **Palestine** so the rest of the pipeline (and the time-series notebook, which already uses "Palestine") line up.

**Outputs of this notebook** — both saved to `data/processed/`:
- `country_features.csv` — the clean numeric feature matrix per country
- `country_clusters.csv` — country, cluster label, and the headline metrics

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


REPO = Path("road-safety-explorer") if Path("road-safety-explorer").exists() else Path(".")
RAW = REPO / "data" / "raw"
PROCESSED = REPO / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

WHO_FILE = RAW / "gsrrs23-indicators-for-participating-countries-or-territories.xlsx"


sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("Pandas :", pd.__version__)
print("Reading from:", WHO_FILE)
print("Writing to  :", PROCESSED)

Pandas : 2.2.2
Reading from: data/raw/gsrrs23-indicators-for-participating-countries-or-territories.xlsx
Writing to  : data/processed


## 1. analyize raw file before i touching it

Before any cleaning. The WHO release is a single spreadsheet with two sheets: `Indicators` (the actual data) and `Notes` (metadata about the columns).

The thing to notice — and the reason we can't just call `pd.read_excel(...)` and walk away — is that the `Indicators` sheet has **two header rows**:

- Row 0 = short variable codes (`CP_3`, `CP_19`, `CP_23` …)
- Row 1 = human-readable descriptions (`Population`, `WHO-estimated road traffic fatalities`, …)
- Row 2 onwards = the actual country data

If we read it normally pandas will treat row 0 as the header and shove row 1 (the descriptions) into the data, which will turn every numeric column into text and quietly break everything. So step one is to read the file *raw* and see the mess for ourselves.

In [6]:
raw = pd.read_excel(WHO_FILE, sheet_name="Indicators", header=None)

print("Raw shape :", raw.shape)
print("Sheets    :", pd.ExcelFile(WHO_FILE).sheet_names)
print()
print("First 3 rows × first 6 columns:")
raw.iloc[:3, :6]

Raw shape : (174, 228)
Sheets    : ['Indicators', 'Notes']

First 3 rows × first 6 columns:


,0,1,2,3,4,5
0,WHO_LEGAL_STATUS_TITLE,CP_0,CP_1,CP_3,CP_5,CP_7
1,WHO status,ISO_3 country name,Country name,Population,Income group,WHO Region
2,Member State,AFG,Afghanistan,40099462,Low income,Eastern Mediterranean Region
